In [1]:
import os 
import sys 
import glob 
import re 
import nltk 
import string 
os.chdir(r'C:\Users\ADMIN\Documents\Nam_3\CS419\project')
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
from pathlib import Path

from source.utils.textPreprocessing import TextPreprocessing
from source.utils.inverted_index import InvertedIndex
from source.utils.BM25_retriever import BM25Retriever
from source.utils.LSA import LSA
from source.utils.Tf_Idf_retriever import TfIDF
from source.utils.evaluations import Evaluator
from source.utils.evaluations import evaluate_many


pd.set_option('display.max_rows', 1000)
pd.set_option("max_colwidth", 40)

OUTPUT_DIR = Path(r"C:\Users\ADMIN\Documents\Nam_3\CS419\project\metrics_analysis\metrics_comparison")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## **DATA PATH**

In [2]:
medline_docs_path = r'C:\Users\ADMIN\Documents\Nam_3\CS419\project\data\medline\new_version\new_medline_docs.csv'
medline_queries_path = r'C:\Users\ADMIN\Documents\Nam_3\CS419\project\data\medline\new_version\new_medline_queries.csv'
medline_qrels_path = r'C:\Users\ADMIN\Documents\Nam_3\CS419\project\data\medline\new_version\new_medline_qrels.csv'

medline_docs_df = pd.read_csv(medline_docs_path)
medline_queries_df = pd.read_csv(medline_queries_path)
medline_qrels_df = pd.read_csv(medline_qrels_path)

ml_docs_df = medline_docs_df[['doc_id', 'doc_text']] 
ml_queries_df = medline_queries_df[['query_id', 'query_text']] 

ml_docs_df.rename(columns={'doc_id':'id', 'doc_text':'content'}, inplace=True) 
ml_queries_df.rename(columns={'query_id':'id', 'query_text':'content'}, inplace=True) 

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_5372\2607394835.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ml_docs_df.rename(columns={'doc_id':'id', 'doc_text':'content'}, inplace=True)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_5372\2607394835.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ml_queries_df.rename(columns={'query_id':'id', 'query_text':'content'}, inplace=True)


In [3]:
cranfield_documents_path = r'C:\Users\ADMIN\Documents\Nam_3\CS419\project\data\cranfield\documents.csv'
cranfield_queries_path = r'C:\Users\ADMIN\Documents\Nam_3\CS419\project\data\cranfield\queries.csv' 
cranfield_qrels_path = r'C:\Users\ADMIN\Documents\Nam_3\CS419\project\data\cranfield\cranfield_qrels.csv'

cranfield_docs_df = pd.read_csv(cranfield_documents_path)
cranfiedl_queries_df = pd.read_csv(cranfield_queries_path)
cranfield_qrels_df = pd.read_csv(cranfield_qrels_path)

## **DEFINE EACH TEXT PREPROCESSOR FOR EACH CORPUS**

In [4]:
medline_tp = TextPreprocessing()

processed_medline_docs = medline_tp.process_dataframe(ml_docs_df)
medline_vocab = medline_tp.get_vocab()

medline_index = InvertedIndex(processed_medline_docs, medline_vocab) 
medline_index._build()

In [5]:
cranfield_tp = TextPreprocessing()

processed_cranfield_docs = cranfield_tp.process_dataframe(cranfield_docs_df)
cranfield_vocab = cranfield_tp.get_vocab()

cranfield_index = InvertedIndex(processed_cranfield_docs, cranfield_vocab) 
cranfield_index._build()

## **DEFINE MODELS FOR EACH CORPUS**

In [6]:
medline_bm25 = BM25Retriever(processed_docs=processed_medline_docs, inverted_index_builder=medline_index) 
medline_tfidf = TfIDF(processed_docs=processed_medline_docs, inverted_index=medline_index) 

medline_lsi = LSA(processed_docs=processed_medline_docs, vocab=medline_vocab, inverted_index_builder=medline_index, dims=200) 
medline_lsi._vectorize(use_tfidf=True)
medline_lsi._fit_svd()

In [7]:
cranfield_bm25 = BM25Retriever(processed_docs=processed_cranfield_docs, inverted_index_builder=cranfield_index) 
cranfield_tfidf = TfIDF(processed_docs=processed_cranfield_docs, inverted_index=cranfield_index) 

cranfield_lsi = LSA(processed_docs=processed_cranfield_docs, vocab=cranfield_vocab, inverted_index_builder=cranfield_index, dims=200) 
cranfield_lsi._vectorize(use_tfidf=True)
cranfield_lsi._fit_svd()

## **Formalize Queries and Qrels**

In [8]:
def get_first_existing(row, candidates):
    for col in candidates:
        if col in row and row[col] is not None and not pd.isna(row[col]):
            return row[col]
    return None


def build_qrels_dict(qrels_records):
    qrels_dict = {}

    for row in qrels_records:
        qid = get_first_existing(row, ["query_id", "qid", "queryid"])
        did = get_first_existing(row, ["doc_id", "docid", "document_id"])
        rel = get_first_existing(row, ["relevance", "rel", "score", "label"])

        if qid is None or did is None or rel is None:
            continue

        qrels_dict.setdefault(qid, {})[did] = float(rel)

    return qrels_dict

In [9]:
medline_queries_records = ml_queries_df.to_dict(orient="records")
medline_qrels_records = medline_qrels_df.to_dict(orient='records')

cranfield_queries_records = cranfiedl_queries_df.to_dict(orient="records") 
cranfield_qrels_records = cranfield_qrels_df.to_dict(orient='records')

medline_qrels_records = build_qrels_dict(medline_qrels_records) 
cranfield_qrels_records = build_qrels_dict(cranfield_qrels_records)

## **Experiment**

In [10]:
corpora = {
    "cranfield":{
        "docs":processed_cranfield_docs, 
        "queries":cranfield_queries_records, 
        "qrels":cranfield_qrels_records
    }, 
    "medline":{
        "docs":processed_medline_docs, 
        "queries":medline_queries_records, 
        "qrels":medline_qrels_records
    }
}


models_by_corpus = {
    "cranfield": {
        "tfidf": cranfield_tfidf,
        "bm25": cranfield_bm25,
        "lsi": cranfield_lsi,
    },
    "medline": {
        "tfidf": medline_tfidf,
        "bm25": medline_bm25,
        "lsi": medline_lsi,
    }
}

K_VALUES = [5, 15, 25, 50]

In [11]:
all_summary_rows = []

for corpus_name, corpus in corpora.items():
    print(f"\n===== Evaluating corpus: {corpus_name.upper()} =====")

    corpus_output_dir = OUTPUT_DIR / corpus_name
    corpus_output_dir.mkdir(parents=True, exist_ok=True)

    queries = corpus.get('queries', None)
    qrels = corpus.get('qrels', None)

    models = models_by_corpus[corpus_name]

    for k in K_VALUES:
        print(f"\n--- top_k = {k} ---")

        results = evaluate_many(
            models=models,
            queries=queries,
            qrels=qrels,
            top_k=k,
            relevance_mode="direct",
            return_results=False,
            return_df=True,
            silent_errors=False,
        )

        for model_name, eval_result in results.items():
            summary = eval_result.summary.copy()

            summary_row = {
                "corpus": corpus_name,
                "model": model_name,
                "k": k,
                **summary,
            }

            all_summary_rows.append(summary_row)
            print(f"\n[{corpus_name.upper()}] {model_name.upper()} @ {k}")
            display(pd.DataFrame([summary_row]))

            per_query_df = eval_result.per_query_df.copy()
            per_query_df = per_query_df.reset_index()

            per_query_path = corpus_output_dir / f"{model_name}_{k}_per_query_df.csv"
            per_query_df.to_csv(per_query_path, index=False, encoding="utf-8-sig")

            print(f"Saved: {per_query_path}")


===== Evaluating corpus: CRANFIELD =====

--- top_k = 5 ---

[CRANFIELD] TFIDF @ 5


,corpus,model,k,Precision@5,Recall@5,F1@5,MAP@5,Ndcg@5
0,cranfield,tfidf,5,0.289778,0.261836,0.244571,0.249862,0.280142


Saved: C:\Users\ADMIN\Documents\Nam_3\CS419\project\metrics_analysis\metrics_comparison\cranfield\tfidf_5_per_query_df.csv

[CRANFIELD] BM25 @ 5


,corpus,model,k,Precision@5,Recall@5,F1@5,MAP@5,Ndcg@5
0,cranfield,bm25,5,0.320889,0.294083,0.274449,0.263564,0.305356


Saved: C:\Users\ADMIN\Documents\Nam_3\CS419\project\metrics_analysis\metrics_comparison\cranfield\bm25_5_per_query_df.csv

[CRANFIELD] LSI @ 5


...

Saved: C:\Users\ADMIN\Documents\Nam_3\CS419\project\metrics_analysis\metrics_comparison\medline\tfidf_50_per_query_df.csv

[MEDLINE] BM25 @ 50


,corpus,model,k,Precision@50,Recall@50,F1@50,MAP@50,Ndcg@50
0,medline,bm25,50,0.594,0.41658,0.376799,0.610782,0.493971


Saved: C:\Users\ADMIN\Documents\Nam_3\CS419\project\metrics_analysis\metrics_comparison\medline\bm25_50_per_query_df.csv

[MEDLINE] LSI @ 50


,corpus,model,k,Precision@50,Recall@50,F1@50,MAP@50,Ndcg@50
0,medline,lsi,50,0.5372,0.319336,0.320976,0.516126,0.456645


Saved: C:\Users\ADMIN\Documents\Nam_3\CS419\project\metrics_analysis\metrics_comparison\medline\lsi_50_per_query_df.csv


In [13]:
summary_metrics_df = pd.DataFrame(all_summary_rows)

summary_path = OUTPUT_DIR / "summary_metrics.csv"
summary_metrics_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

display(summary_metrics_df)

print(f"Saved summary metrics to: {summary_path}")

,corpus,model,k,Precision@5,Recall@5,F1@5,MAP@5,Ndcg@5,Precision@15,Recall@15,...,Precision@25,Recall@25,F1@25,MAP@25,Ndcg@25,Precision@50,Recall@50,F1@50,MAP@50,Ndcg@50
0,cranfield,tfidf,5,0.289778,0.261836,0.244571,0.249862,0.280142,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,cranfield,bm25,5,0.320889,0.294083,0.274449,0.263564,0.305356,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,cranfield,lsi,5,0.314667,0.283335,0.265753,0.271994,0.310833,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,cranfield,tfidf,15,NaN,NaN,NaN,NaN,NaN,0.184000,0.448443,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,cranfield,bm25,15,NaN,NaN,NaN,NaN,NaN,0.186667,0.452068,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,cranfield,lsi,15,NaN,NaN,NaN,NaN,NaN,0.195259,0.476222,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,cranfield,tfidf,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.132267,0.521405,0.196287,0.256840,0.371372,NaN,NaN,NaN,NaN,NaN
7,cranfield,bm25,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.136889,0.537265,0.203297,0.267682,0.386023,NaN,NaN,NaN,NaN,NaN
8,cranfield,lsi,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.145600,0.578196,0.216847,0.289118,0.409694,NaN,NaN,NaN,NaN,NaN
9,cranfield,tfidf,50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.081956,0.629733,0.138196,0.268726,0.402423


Saved summary metrics to: C:\Users\ADMIN\Documents\Nam_3\CS419\project\metrics_analysis\metrics_comparison\summary_metrics.csv
